# What Is a Tensor?

A **tensor** is PyTorch's fundamental data structure for representing numerical data.

At its simplest, a tensor is a container of numbers arranged along one or more dimensions.

You can think of tensors as a generalization of familiar mathematical objects:

* A single number → **0-dimensional tensor**
* A vector → **1-dimensional tensor**
* A matrix → **2-dimensional tensor**
* Higher-dimensional numerical data → **3D, 4D, ... tensors**

For example:

```text
Scalar:

5


Vector:

[1, 2, 3]


Matrix:

[[1, 2, 3],
 [4, 5, 6]]


3D Tensor:

[
    [[1, 2],
     [3, 4]],

    [[5, 6],
     [7, 8]]
]
```

The important idea is that a tensor is not simply a Python list.

PyTorch tensors are designed specifically for numerical computing and deep learning. They support:

* Efficient numerical operations
* GPU acceleration
* Automatic differentiation
* Different numerical data types
* Operations across multiple dimensions

This makes tensors the basic building blocks on which PyTorch models and training systems are built.


In [20]:
import torch

## Creating Our First Tensor

Let's create a simple tensor containing three numbers.


In [21]:
x = torch.tensor([1, 2, 3])

print(x)
print(type(x))

tensor([1, 2, 3])
<class 'torch.Tensor'>


## Tensor Metadata

A tensor contains more than just numerical values. PyTorch also tracks important metadata describing how those values are represented and where they are stored.

Four particularly important properties are:

| Property | Meaning                                  |
| -------- | ---------------------------------------- |
| `ndim`   | Number of dimensions (axes)              |
| `shape`  | Size of the tensor along each dimension  |
| `dtype`  | Numerical data type of the elements      |
| `device` | Device where the tensor's data is stored |

Let's inspect them:


In [22]:
print(x.ndim)
print(x.shape)
print(x.dtype)
print(x.device)

1
torch.Size([3])
torch.int64
cpu


### Example: A 2D Tensor

For a matrix with shape `(3, 4)`:

```text
3 → number of rows
4 → number of columns
```

Therefore:

* `ndim = 2`
* `shape = (3, 4)`

For higher-dimensional tensors, `shape` contains one value for each axis rather than simply representing rows and columns.


## References and Memory

Python variables do not necessarily contain an independent copy of an object. Assigning one variable to another can make both variables refer to the **same object**.

This matters for tensors because copying large amounts of numerical data can be expensive in both memory and computation.


In [23]:
x = torch.tensor([1, 2, 3])
y = x

print(id(x))
print(id(y))
print(x)
print(y)

x[0] = 100

print(x)
print(y)

131430019677520
131430019677520
tensor([1, 2, 3])
tensor([1, 2, 3])
tensor([100,   2,   3])
tensor([100,   2,   3])


### Reassignment Is Different

Changing a tensor and reassigning a variable are different operations.

When we modify the tensor through indexing, both references still point to the same object.

However, if we reassign `x` to a completely new tensor, `y` still refers to the original tensor.


In [24]:
x = torch.tensor([1, 2, 3])
y = x

x = torch.tensor([100, 2, 3])

print(x)
print(y)

tensor([100,   2,   3])
tensor([1, 2, 3])


## Creating an Independent Copy

Assigning a tensor to another variable does not create an independent copy:

```python
y = x
```

Both variables refer to the same tensor.

When an independent copy is required, PyTorch provides `clone()`.

A cloned tensor has its **own storage**, so modifying one tensor does not modify the other.


In [25]:
x = torch.tensor([1, 2, 3])
y = x.clone()

x[0] = 100

print(x)
print(y)

tensor([100,   2,   3])
tensor([1, 2, 3])


### Views Share Storage

A **view** is a tensor that provides a different way of looking at the same underlying storage.

Therefore:

* The original tensor and the view are different tensor objects.
* They can have different shapes or indexing.
* They share the same underlying storage.
* Modifying the shared data through one tensor can affect the other.

This is different from `clone()`, which creates independent storage.


In [26]:
x = torch.tensor([1, 2, 3, 4])
y = x[1:3]

print(x)
print(y)

y[0] = 100

print(x)
print(y)

tensor([1, 2, 3, 4])
tensor([2, 3])
tensor([  1, 100,   3,   4])
tensor([100,   3])


In [27]:
print(id(x) == id(y))
print(x.storage().data_ptr() == y.storage().data_ptr())

False
True


### `reshape()`

`reshape()` changes the shape of a tensor without changing its number of elements.

When the existing memory layout allows it, PyTorch can implement the reshape as a **view**, avoiding a data copy.

However, `reshape()` does **not** guarantee that a view will always be possible. If the requested shape cannot be represented using the existing memory layout, PyTorch may create a copy instead.

Therefore:

> `reshape()` can return a view when possible, but you should not treat it as a guaranteed view operation.


# Indexing and Slicing

PyTorch tensors support Python-style indexing and slicing.

Indexing selects individual elements, while slicing selects ranges of elements.

The basic pattern is:

```python
tensor[start:stop:step]
```

As with Python sequences, the `stop` index is **exclusive**.

For multidimensional tensors, we can provide one index or slice for each dimension.


### Slicing

Slicing selects a range of elements along one or more dimensions.

The basic syntax is:

```python id="s9c3ue"
start:stop:step
```

where:

* `start` is inclusive
* `stop` is exclusive
* `step` determines the spacing between selected elements

For a 2D tensor, we can slice each dimension independently.

For example:

```python id="x[0:2, :]"
```

means:

* rows `0` through `1`
* all columns


In [28]:
x = torch.tensor([
    [10, 20, 30],
    [40, 50, 60],
    [70, 80, 90]
])

print(x)
print(x.shape)

tensor([[10, 20, 30],
        [40, 50, 60],
        [70, 80, 90]])
torch.Size([3, 3])


In [29]:
print(x[0:2, :])
print(x[1:, 0:2])
print(x[::2, ::2])

tensor([[10, 20, 30],
        [40, 50, 60]])
tensor([[40, 50],
        [70, 80]])
tensor([[10, 30],
        [70, 90]])


### Indexing Higher-Dimensional Tensors

The same indexing principles apply to tensors with more than two dimensions.

For a tensor with shape `(2, 3, 4)`:

- Dimension 0 has size 2
- Dimension 1 has size 3
- Dimension 2 has size 4

Each index selects one position along its corresponding dimension.

In [30]:
images = torch.arange(24).reshape(2, 3, 4)

print(images)
print(images.shape)

tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11]],

        [[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]])
torch.Size([2, 3, 4])


In [31]:
print(images[1, 2, 3])

tensor(23)


For `images[1, 2, 3]`:

- `1` selects the second block
- `2` selects the third row
- `3` selects the fourth element

Indexing always starts from `0`.

## Indexing and Slicing — Summary

- `tensor[i]` selects along the first dimension.
- `tensor[i, j]` selects along the first two dimensions.
- `:` selects all elements along a dimension.
- `start:stop` selects a range, with `stop` excluded.
- `start:stop:step` allows control over the step.
- Negative indices count from the end.
- The same indexing and slicing principles extend to tensors of any number of dimensions.

In [32]:
# Quick reference

print(x[1])        # Second row
print(x[:, 1])     # Second column
print(x[0:2, :])   # First two rows
print(x[1:, 0:2])  # Last two rows, first two columns
print(x[::2, ::2]) # Every second row and column

tensor([40, 50, 60])
tensor([20, 50, 80])
tensor([[10, 20, 30],
        [40, 50, 60]])
tensor([[40, 50],
        [70, 80]])
tensor([[10, 30],
        [70, 90]])


## Tensor Operations

PyTorch supports element-wise operations directly on tensors.

For element-wise operations, corresponding elements are operated on independently.

In [33]:
a = torch.tensor([1, 2, 3])
b = torch.tensor([4, 5, 6])

print(a + b)
print(a - b)
print(a * b)
print(a / b)

tensor([5, 7, 9])
tensor([-3, -3, -3])
tensor([ 4, 10, 18])
tensor([0.2500, 0.4000, 0.5000])


### Element-wise Multiplication

The `*` operator performs element-wise multiplication.

For example:

```text
[1, 2, 3] * [4, 5, 6]

= [1×4, 2×5, 3×6]

= [4, 10, 18]

## Matrix Multiplication

The `@` operator performs matrix multiplication.

For matrices `A` and `B`, the inner dimensions must match:

```text
(m, n) @ (n, p) → (m, p)

In [35]:
A = torch.tensor([
    [1, 2],
    [3, 4]
])

B = torch.tensor([
    [5, 6],
    [7, 8]
])

print(A @ B)

tensor([[19, 22],
        [43, 50]])


For the matrices above:

```text
[1 2]   [5 6]
[3 4] × [7 8]

In [36]:
print(A.shape)
print(B.shape)
print((A @ B).shape)

torch.Size([2, 2])
torch.Size([2, 2])
torch.Size([2, 2])


### Element-wise vs Matrix Multiplication

| Operation | Operator | Meaning |
|---|---|---|
| Addition | `+` | Element-wise |
| Subtraction | `-` | Element-wise |
| Multiplication | `*` | Element-wise |
| Matrix multiplication | `@` | Linear algebra matrix product |

Understanding this distinction is essential because neural networks rely heavily on matrix multiplication.

## Broadcasting

Broadcasting allows PyTorch to perform operations on tensors with different shapes when their dimensions are compatible.

PyTorch compares dimensions from **right to left**.

Two dimensions are compatible when:

1. They are equal, or
2. One of them is `1`.

Missing leading dimensions are treated as `1`.

The resulting size of each compatible dimension is the larger of the two sizes.

In [37]:
a = torch.tensor([
    [1],
    [2],
    [3]
])

b = torch.tensor([
    [10, 20, 30]
])

print(a.shape)
print(b.shape)

c = a + b

print(c)
print(c.shape)

torch.Size([3, 1])
torch.Size([1, 3])
tensor([[11, 21, 31],
        [12, 22, 32],
        [13, 23, 33]])
torch.Size([3, 3])


Here:

```text
a.shape = (3, 1)
b.shape = (1, 3)

In [38]:
a = torch.tensor([
    [1, 2],
    [3, 4]
])

b = torch.tensor([
    [10, 20, 30]
])

print(a.shape)
print(b.shape)

# This operation will raise an error because
# the dimensions 2 and 3 are incompatible.
try:
    print(a + b)
except RuntimeError as e:
    print("RuntimeError:", e)

torch.Size([2, 2])
torch.Size([1, 3])
RuntimeError: The size of tensor a (2) must match the size of tensor b (3) at non-singleton dimension 1


### Broadcasting Algorithm

For two tensor shapes:

1. Align the shapes from the right.
2. Add leading `1`s to the shorter shape if necessary.
3. Compare each dimension.
4. If the dimensions are equal or one is `1`, broadcasting is possible.
5. Otherwise, the operation is not possible.
6. The output dimension is the larger compatible dimension.

Example:

```text
(8, 1, 6, 1)
(   7, 1, 5)


### Markdown cell

```markdown id="p7k3w1"
### Broadcasting and Memory

Broadcasting does not necessarily create a physically replicated copy of the data.

It allows PyTorch to treat dimensions of size `1` as if their values were repeated where needed.

This makes broadcasting both convenient and memory-efficient.

In [39]:
# Quick broadcasting examples

x = torch.tensor([[1], [2], [3]])
y = torch.tensor([[10, 20, 30]])

print(x + y)

tensor([[11, 21, 31],
        [12, 22, 32],
        [13, 23, 33]])


## CPU and GPU

PyTorch tensors can be stored and processed on different devices.

The most common devices are:

- `cpu` — system memory and CPU computation
- `cuda` — NVIDIA GPU memory and GPU computation

PyTorch creates tensors on the CPU by default unless another device is specified.

GPUs are particularly effective for deep learning because they can perform large numbers of similar, independent numerical operations in parallel.

In [40]:
x = torch.tensor([1, 2, 3])

print(x.device)

cpu


### Moving a Tensor to a GPU

If CUDA is available, a tensor can be moved to the GPU with `.to()`.

The original CPU tensor and the GPU tensor occupy different device memory.

In [41]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

x = torch.tensor([1, 2, 3])
x_device = x.to(device)

print("CPU tensor:", x.device)
print("Selected device:", x_device.device)

CPU tensor: cpu
Selected device: cpu


### Device Compatibility

Operations involving multiple tensors generally require the tensors to be on the same device.

PyTorch does not silently move tensors between CPU and GPU during an operation.

This avoids unexpected data transfers and makes device movement explicit.

In [42]:
if torch.cuda.is_available():
    x = torch.tensor([1, 2, 3])
    y = torch.tensor([4, 5, 6], device="cuda")

    try:
        print(x + y)
    except RuntimeError as e:
        print("RuntimeError:", e)
else:
    print("CUDA is not available on this system.")

CUDA is not available on this system.


### Moving Data Between Devices

A tensor on the GPU can be moved back to the CPU with `.cpu()`.

```python
x_cpu = x_gpu.cpu()

In [43]:
x = torch.tensor([1, 2, 3])

if torch.cuda.is_available():
    x_gpu = x.to("cuda")
    x_cpu = x_gpu.cpu()

    print(x_gpu.device)
    print(x_cpu.device)
else:
    print("CUDA is not available.")

CUDA is not available.


## 10. NumPy ↔ PyTorch Interoperability

PyTorch provides convenient interoperability with NumPy.

For CPU tensors, PyTorch and NumPy can share the same underlying memory. This avoids an unnecessary data copy.

`torch.from_numpy()` creates a tensor that shares memory with the NumPy array.

In [44]:
import numpy as np

a = np.array([1, 2, 3])
t = torch.from_numpy(a)

print(a)
print(t)

[1 2 3]
tensor([1, 2, 3])


In [45]:
a[0] = 100

print(a)
print(t)

[100   2   3]
tensor([100,   2,   3])


Because the NumPy array and PyTorch tensor share memory, modifying one can affect the other.

The same applies in the opposite direction.

In [46]:
t[1] = 999

print(a)
print(t)

[100 999   3]
tensor([100, 999,   3])


### Converting a Tensor to NumPy

A CPU tensor can be converted to a NumPy array using `.numpy()`.

When possible, the resulting NumPy array shares the tensor's underlying memory.

In [47]:
t = torch.tensor([1, 2, 3])
a = t.numpy()

print(t)
print(a)

t[0] = 100

print(t)
print(a)

tensor([1, 2, 3])
[1 2 3]
tensor([100,   2,   3])
[100   2   3]


### GPU Tensors and NumPy

NumPy operates on CPU memory and cannot directly access GPU memory.

Therefore, a CUDA tensor must first be moved to the CPU before converting it to NumPy.

```python
numpy_array = gpu_tensor.cpu().numpy()

In [48]:
if torch.cuda.is_available():
    gpu_tensor = torch.tensor([1, 2, 3], device="cuda")

    cpu_tensor = gpu_tensor.cpu()
    numpy_array = cpu_tensor.numpy()

    print(gpu_tensor.device)
    print(cpu_tensor.device)
    print(numpy_array)
else:
    print("CUDA is not available.")

CUDA is not available.


### Summary

| Operation | Memory behavior |
|---|---|
| `torch.from_numpy(array)` | Shares memory when supported |
| `tensor.numpy()` | Shares memory when supported |
| `tensor.cpu()` | Moves data to CPU memory |
| `gpu_tensor.numpy()` | Not allowed directly |

The important distinction is between **CPU memory** and **GPU memory**. NumPy works with CPU memory, while CUDA tensors store their data in GPU memory.

# Exercises

Try to solve each exercise before running the code.

## Exercise 1 — Tensor Metadata

Create:

```python
x = torch.tensor([[1, 2, 3], [4, 5, 6]])
```

Print:

* `ndim`
* `shape`
* `dtype`
* `device`

---

## Exercise 2 — Indexing

Given:

```python
x = torch.tensor([
    [10, 20, 30],
    [40, 50, 60],
    [70, 80, 90]
])
```

Predict the output of:

```python
x[2, 1]
x[0]
x[:, 2]
```

---

## Exercise 3 — Slicing

Predict the output of:

```python
x[0:2, 1:]
x[::2, ::2]
```

---

## Exercise 4 — Broadcasting

Determine whether each operation is valid and predict the output shape.

```text
(3, 1) + (1, 4)
(5, 2) + (5, 4)
(2, 3, 4) + (3, 4)
(2, 3, 4) + (2, 4)
```

For each pair, apply the broadcasting rules:

1. Align dimensions from the right.
2. Add leading `1`s to the shorter shape if necessary.
3. Dimensions must either be equal or one of them must be `1`.

---

## Exercise 5 — Memory

Predict whether the tensors share storage.

### Case A

```python
x = torch.tensor([1, 2, 3])
y = x
```

### Case B

```python
x = torch.tensor([1, 2, 3])
y = x.clone()
```

### Case C

```python
x = torch.tensor([1, 2, 3, 4])
y = x[1:3]
```

---

## Exercise 6 — Reshape

Predict whether modifying `y` can modify `x`.

```python
x = torch.tensor([1, 2, 3, 4, 5, 6])
y = x.reshape(2, 3)

y[0, 0] = 100
```

---

## Exercise 7 — NumPy Interoperability

Predict the value of `t` after the following code:

```python
import numpy as np

a = np.array([1, 2, 3])
t = torch.from_numpy(a)

a[0] = 100
```

---

## Exercise 8 — Devices

Why does this operation fail when CUDA is available?

```python
x = torch.tensor([1, 2, 3])
y = torch.tensor([4, 5, 6], device="cuda")

x + y
```

What must be done before the operation can succeed?

---

## Challenge — Broadcasting

Without running the code, predict the output shape:

```python
a = torch.randn(8, 1, 6, 1)
b = torch.randn(7, 1, 5)

result = a + b
```

Explain the broadcasting decision dimension by dimension.

---

## Final Challenge

Predict the output of each operation:

```python
x = torch.tensor([
    [10, 20, 30],
    [40, 50, 60],
    [70, 80, 90]
])

print(x[1:, :2])
print(x[:, ::2])
print(x[::2, 1:])
```

Then verify your predictions by running the code.
